# Objetivo 3 — Interações metabólicas entre leveduras vínicas (SMETANA)

**Projeto:** STrengthS4WineChaiN · TASK 3.1 — consórcios microbianos para vinho com menor teor alcoólico

Construção das comunidades, validação estrutural e cálculo dos índices de interação do SMETANA para os quatro
modelos curados nos objetivos 1a e 1b, no mosto sintético MS300 definido no objetivo 2.

| código | espécie | ficheiro curado | regime |
|---|---|---|---|
| Sc | *Saccharomyces cerevisiae* | `yeast-GEM_curado.xml` | anaerobiose |
| Td | *Torulaspora delbrueckii* | `model_thermotolerans_curado.xml` | anaerobiose |
| Mp | *Metschnikowia pulcherrima* | `model_pulcherrima_curado.xml` | microaerobiose, q(O₂) = 0.5 |
| Lt | *Lachancea thermotolerans* | `model_delbrueckii_curado.xml` | anaerobiose |

**O passo que este notebook tem de resolver antes de qualquer índice.** Os quatro modelos não partilham
espaço de nomes no compartimento extracelular. O yeast-GEM identifica a glucose como `s_0565` e os três
CarveFungi como `glcD_e`. O SMETANA agrupa o meio partilhado pelo identificador do metabolito, por isso,
sem reconciliação, a glucose da *S. cerevisiae* e a glucose da *T. delbrueckii* são dois compartimentos
distintos. A secção 2 mede o efeito: **zero** metabolitos extracelulares em comum entre a *S. cerevisiae* e
qualquer não-*Saccharomyces*. Os índices calculados nesse estado não estão errados por pouco — estão
estruturalmente vazios, exatamente na classe de interação que este projeto existe para medir.

O notebook de referência do projeto das leguminosas resolvia um problema da mesma família
(`harmonize_koduru_ids`, os `__45__`), mas ali era codificação de caracteres no mesmo espaço de nomes BiGG.
Aqui são dois espaços de nomes diferentes e a reconciliação tem de ser feita por anotação química.

## 1 · Preparação

In [1]:
import os, sys, time, itertools, warnings
import pandas as pd
import numpy as np
warnings.filterwarnings('ignore')

RAIZ = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, os.path.join(RAIZ, 'scripts'))

import cobra
import harmonizacao as H
import curadoria_minima as CU
import meio_mosto as MM
import consorcios as CM

from reframed import load_cbmodel, FBA, Environment
from smetana.interface import Community
from smetana.smetana import mip_score, mro_score, sc_score, mu_score, mp_score

_lic = os.path.expanduser("~/gurobi.lic")
if os.path.exists(_lic):
    os.environ["GRB_LICENSE_FILE"] = _lic

CURADOS = {
    'Sc': os.path.join(RAIZ, 'models/curados/yeast-GEM_curado.xml'),
    'Lt': os.path.join(RAIZ, 'models/curados/model_thermotolerans_curado.xml'),
    'Mp': os.path.join(RAIZ, 'models/curados/model_pulcherrima_curado.xml'),
    'Td': os.path.join(RAIZ, 'models/curados/model_delbrueckii_curado.xml'),
}
SAIDA = os.path.join(RAIZ, 'models/harmonizados')
os.makedirs(SAIDA, exist_ok=True)

for cod, p in CURADOS.items():
    if not os.path.exists(p):
        raise FileNotFoundError(f'{cod}: {p}')

# mu solo de referencia, medido no obj2 (MS300, cenario central, regime proprio).
# Serve de teste de aceitacao da harmonizacao na seccao 3.4.
MU_OBJ2 = {'Sc': 0.1654, 'Td': 0.1835, 'Mp': 0.1836, 'Lt': 0.1835}

print(f'cobra {cobra.__version__}')
print(f'Q_O2 por organismo: {CM.Q_O2}')
print(f'mu de referencia do obj2: {MU_OBJ2}')

cobra 0.31.1
Q_O2 por organismo: {'Sc': 0.0, 'Td': 0.0, 'Mp': 0.5, 'Lt': 0.0}
mu de referencia do obj2: {'Sc': 0.1654, 'Td': 0.1835, 'Mp': 0.1836, 'Lt': 0.1835}


## 2 · Reconciliação do compartimento extracelular

### 2.1 · O diagnóstico que justifica a secção

Contagem dos metabolitos extracelulares em comum entre cada par, **antes** de qualquer reconciliação.
É a medida directa do meio partilhado que o SMETANA vai conseguir ver.

In [2]:
M = {cod: cobra.io.read_sbml_model(p) for cod, p in CURADOS.items()}

antes = pd.DataFrame(H.diagnostico_pools(list(M.values()), list(M)))
display(antes)

print('\nComo cada modelo escreve os mesmos compostos:')
for cod in ('Sc', 'Td'):
    linhas = []
    for r in list(M[cod].exchanges)[:6]:
        met = next(iter(r.metabolites))
        linhas.append(f'   troca {r.id:<14} metabolito {met.id:<14} "{met.name[:34]}"')
    print(f' {cod}:'); print('\n'.join(linhas))

Set parameter Username
Set parameter LicenseID to value 2818537
Academic license - for non-commercial use only - expires 2027-05-05


,A,B,ext_A,ext_B,partilhados,jaccard
0,Sc,Lt,279,251,0,0.000000
1,Sc,Mp,279,253,0,0.000000
2,Sc,Td,279,239,0,0.000000
3,Lt,Mp,251,253,225,0.806452
4,Lt,Td,251,239,219,0.808118
5,Mp,Td,253,239,213,0.763441



Como cada modelo escreve os mesmos compostos:
 Sc:
   troca r_1542         metabolito s_0003         "(1->3)-beta-D-glucan"
   troca r_1545         metabolito s_0022         "(R)-carnitine"
   troca r_1546         metabolito s_0026         "(R)-lactate"
   troca r_1547         metabolito s_0029         "(R)-mevalonate"
   troca r_1548         metabolito s_0032         "(R)-pantothenate"
   troca r_1549         metabolito s_0036         "(R,R)-2,3-butanediol"
 Td:
   troca UF02549_E      metabolito ash_1g_e       "ash 1 g"
   troca UF02628_E      metabolito fum_e          "fumarate(2-)"
   troca UF02691_E      metabolito ins_e          "inosine"
   troca UF02726_E      metabolito mtpp_e         "3-(methylthio)propionate"
   troca UF02765_E      metabolito pi_e           "hydrogenphosphate"
   troca UF02967_E      metabolito lacL_e         "(S)-lactate"


### 2.2 · Construção da correspondência

Espaço de nomes de destino: o dos CarveFungi (estilo BiGG, `<id>_e`). Três dos quatro modelos já o usam e
é o que o `reframed` espera com `flavor='bigg'`.

Três chaves independentes, por ordem de prioridade: `bigg.metabolite` (anotação do yeast-GEM),
ChEBI (normalizado — o yeast-GEM escreve `CHEBI:`, os CarveFungi escrevem `ChEBI:`) e KEGG compound.

O nome **não** entra como chave. Diverge em casos que importam (`L-arginine` contra `L-argininium`,
`D-glucose` contra `alpha-D-glucose`) e coincide em casos em que não devia.

As três chaves são calculadas todas, e o desacordo entre elas é devolvido em `conflitos` em vez de ser
resolvido em silêncio pela prioridade. Um conflito é um erro de anotação num dos ficheiros e tem de ser
lido. Na primeira versão desta tabela o teste apanhou uma entrada manual errada da minha parte
(`s_0458` mapeado para água quando é CO₂); é para isso que serve.

In [3]:
rec = H.construir_mapa_extracelular(M['Sc'], [M['Td'], M['Mp'], M['Lt']])

print('\nConflitos entre chaves:', len(rec['conflitos']))
for mid, cand in rec['conflitos']:
    print(f"   {mid}  {M['Sc'].metabolites.get_by_id(mid).name}  ->  {cand}")

print('\nCorrespondencias manuais (CURADAS), com razao:')
for k, v in H.CURADAS.items():
    print(f"   {k:8s} -> {v:12s}  {M['Sc'].metabolites.get_by_id(k).name}")

print('\nDeixadas deliberadamente por fazer (POR_VERIFICAR):')
for k, (v, razao) in H.POR_VERIFICAR.items():
    print(f'   {k} -> {v}')
    print(f'      {razao}')

fonte, metabolitos extracelulares : 279
destino, universo                 : 286
correspondidos                    : 167
   por curada                     : 3
   por bigg                       : 122
   por chebi                      : 31
   por kegg                       : 11
sem par (so na fonte)             : 112
orfaos  (so no destino)           : 119
conflitos entre chaves            : 0

Conflitos entre chaves: 0

Correspondencias manuais (CURADAS), com razao:
   s_1295   -> 9Zhxda_e      palmitoleate
   s_2826   -> 9Zocda_e      oleate
   s_4242   -> 3mppal_e      methional

Deixadas deliberadamente por fazer (POR_VERIFICAR):
   s_4249 -> 4hpheetoh_e
      tirosol = 4-(2-hidroxietil)fenol; o metabolito CarveFungi chama-se 4-(1-hidroxietil)fenol. Mesma formula, posicao do hidroxilo diferente. Um dos dois nomes esta errado no ficheiro e nao ha anotacao que decida. Por confirmar na reacao que o produz antes de juntar os pools.
   s_4045 -> dtmp_e
      timidina 3'-monofosfato; o yeas

### 2.3 · O que ficou de fora, e porquê

Duas listas distintas, e a diferença entre elas importa.

**Sem par (só no yeast-GEM).** Compostos que a *S. cerevisiae* troca e que nenhum CarveFungi tem no
extracelular. Dois subgrupos com estatuto diferente:

- *Minerais individuais* (K, Na, Ca, Mn, Cu, Zn, Mg). Os CarveFungi não têm trocas minerais separadas:
  têm um pseudo-metabolito agregado, `ash 1 g`, sem fórmula. Não são o mesmo objecto e juntá-los criaria um
  pool partilhado sem sentido físico. Fica declarado: **no eixo mineral os quatro modelos não são
  comparáveis**, e o SMETANA não vê competição nem partilha de minerais entre a *S. cerevisiae* e as
  não-*Saccharomyces*.
- *Compostos de aroma de Ehrlich* (2-feniletanol, 3-metilbutanal). São ausências reais nos ficheiros
  CarveFungi, já declaradas no anexo do `obj1b`. Consequência para este notebook: a via de Ehrlich é uma
  classe de interação que o SMETANA aqui **não consegue ver**, e isso tem de constar das limitações.

**Órfãos (só nos CarveFungi).** 119 compostos, a maioria com carbono. São capacidade de secreção que os
modelos automáticos trazem e o yeast-GEM não tem. Entram nos índices entre as não-*Saccharomyces* e ficam
invisíveis à *S. cerevisiae*, o que enviesa qualquer comparação entre um par com Sc e um par sem Sc.

In [4]:
sem_par = pd.DataFrame([
    {'id': mid,
     'nome': M['Sc'].metabolites.get_by_id(mid).name,
     'formula': M['Sc'].metabolites.get_by_id(mid).formula}
    for mid in rec['sem_par']
])
print(f"Sem par (so no yeast-GEM): {len(sem_par)}")
display(sem_par.head(30))

alvo_mets = {m.id: m for mm in (M['Td'], M['Mp'], M['Lt']) for m in mm.metabolites if m.compartment == 'e'}
orfaos = pd.DataFrame([
    {'id': i, 'nome': alvo_mets[i].name, 'formula': alvo_mets[i].formula}
    for i in rec['orfaos'] if i in alvo_mets
])
print(f"\nOrfaos (so nos CarveFungi): {len(orfaos)}")
display(orfaos.head(20))

Sem par (so no yeast-GEM): 112


,id,nome,formula
0,s_0022,(R)-carnitine,C7H15NO3
1,s_0080,1-(sn-glycero-3-phospho)-1D-myo-inositol,C9H18O11P
2,s_0084,1-acylglycerophosphocholine,C9H20NO7PR
3,s_0186,2-phenylethanol,C8H10O
4,s_0235,3-methylbutanal,C5H10O
5,s_0320,5-formyltetrahydrofolic acid,C20H21N7O7
6,s_0446,bicarbonate,CHO3
7,s_0549,D-arabinose,C5H10O5
8,s_0560,D-galacturonate,C6H9O7
9,s_0962,L-arabinitol,C5H12O5



Orfaos (so nos CarveFungi): 119


,id,nome,formula
0,10Ehpda_e,(10E)-heptadecenoate,C17H31O2
1,10Zhpda_e,(10Z)-heptadecenoate,C17H31O2
2,124bztriol_e,"benzene-1,2,4-triol",C6H6O3
3,25bishydroxymethylfuran_e,"2,5-furandimethanol",C6H8O3
4,2dr5p_e,2-deoxy-D-ribose 5-phosphate,C5H9O7P
5,2hymeph_e,salicyl alcohol,C7H8O2
6,2hyoxplac_e,(2-hydroxyphenyl)acetate,C8H7O3
7,2obut_e,2-oxobutanoate,C4H5O3
8,34dhbz_e,"3,4-dihydroxybenzoate",C7H5O4
9,34dmbza_e,veratric acid,C9H10O4


### 2.4 · Um detalhe de implementação que corrompe os relatórios

O SMETANA recupera o nome do composto a partir do identificador do metabolito com
`original_metabolite[2:-2]`: tira `M_` da frente e **dois caracteres do fim**, assumindo o sufixo `_e`.

Os identificadores do yeast-GEM não trazem o compartimento (`s_1374`, e não `s_1374_e`). Sem sufixo, a
etiqueta sai truncada e, pior, vários metabolitos colapsam na mesma: `s_4199`, `s_4200` e `s_4201` saem
todos como `s_42`. Numa primeira execução deste notebook o meio mínimo aparecia com `s_41` e `s_42` em vez
de sete minerais distintos.

A `aplicar_mapa` acrescenta `_e` a todo o extracelular que ficou sem par, para que a convenção do SMETANA
se aplique sem perda. Não é cosmética: sem isto, os relatórios de meio mínimo são ilegíveis e têm
compostos fundidos.

In [5]:
n_met, n_tr = H.aplicar_mapa(M['Sc'], rec['mapa'], verbose=True)
for cod in ('Td', 'Mp', 'Lt'):
    H.normalizar_trocas(M[cod])

depois = pd.DataFrame(H.diagnostico_pools(list(M.values()), list(M)))
comparacao = antes[['A', 'B', 'partilhados']].merge(
    depois[['A', 'B', 'partilhados', 'jaccard']], on=['A', 'B'], suffixes=('_antes', '_depois'))
display(comparacao)

if comparacao.loc[comparacao.A == 'Sc', 'partilhados_depois'].min() < 100:
    raise RuntimeError('Reconciliacao insuficiente entre o yeast-GEM e os CarveFungi.')
print('Trocas depois da normalizacao:')
for cod in ('Sc', 'Td'):
    print(f"   {cod}: {[r.id for r in list(M[cod].exchanges)[:4]]}")

   167 metabolitos renomeados, 112 sufixados com _e, 270 trocas renomeadas


,A,B,partilhados_antes,partilhados_depois,jaccard
0,Sc,Lt,0,156,0.417112
1,Sc,Mp,0,147,0.381818
2,Sc,Td,0,148,0.400000
3,Lt,Mp,225,225,0.806452
4,Lt,Td,219,219,0.808118
5,Mp,Td,213,213,0.763441


Trocas depois da normalizacao:
   Sc: ['EX_13BDglcn_e', 'EX_s_0022_e', 'EX_lacD_e', 'EX_mevR_e']
   Td: ['EX_ash_1g_e', 'EX_fum_e', 'EX_ins_e', 'EX_mtpp_e']


### 2.5 · Os compostos do mosto sobreviveram à reconciliação?

O critério de aceitação não é a contagem total: é se os compostos que definem o MS300 ficaram todos com um
pool comum. Um mosto reconciliado a 90 % mas sem glucose partilhada não serve para nada.

In [6]:
mapa_met, divergencias = CM.mapa_especie_metabolito(M)
if divergencias:
    print('DIVERGENCIAS entre modelos para o mesmo composto:'); print(divergencias)

comp = MM.composicao_mmol_L()
linhas = []
for especie in sorted(comp):
    met = mapa_met.get(especie)
    presentes = [cod for cod in M if met and met in M[cod].metabolites]
    linhas.append({'especie': especie, 'metabolito': met or '—',
                   'modelos': ''.join(c if c in presentes else '·' for c in ('Sc','Td','Mp','Lt')),
                   'n': len(presentes)})
tab = pd.DataFrame(linhas).sort_values(['n', 'especie'])
display(tab)

partilhados = (tab.n == 4).sum()
print(f'\nCompostos do MS300 presentes nos quatro modelos: {partilhados}/{len(tab)}')
print(f'Presentes em nenhum ou so num: {(tab.n <= 1).sum()}')
print(f"O2 -> {mapa_met.get('oxygen')} | ash -> {mapa_met.get('ash')}")

,especie,metabolito,modelos,n
5,borate,—,····,0
9,cobalt,—,····,0
19,iodide,—,····,0
27,molybdate,—,····,0
6,calcium,s_4199_e,Sc···,1
7,chloride,s_4200_e,Sc···,1
10,copper,s_4201_e,Sc···,1
23,magnesium,s_4204_e,Sc···,1
25,manganese,s_4202_e,Sc···,1
34,potassium,s_1374_e,Sc···,1



Compostos do MS300 presentes nos quatro modelos: 33/45
Presentes em nenhum ou so num: 12
O2 -> o2_e | ash -> ash_1g_e


## 3 · Regime, exportação e teste de aceitação

### 3.1 · A anaerobiose tem de ser aplicada antes de exportar

Os ficheiros `*_curado.xml` são a variante **aeróbia**. A anaerobiose é uma condição de simulação aplicada
por função (`curadoria_minima.modo_anaerobio`) e não existe em ficheiro nenhum. Se o notebook exportasse os
modelos sem a aplicar, o SMETANA correria quatro leveduras aeróbias num mosto.

**A NS1 tem de ser pedida explicitamente.** A `modo_anaerobio` com `ids=None` aplica só as edições marcadas
`aplicar_por_omissao`; as que criam capacidade nova ficam de fora por desenho, para a decisão ficar escrita
aqui e não escondida num valor por omissão. A NS1 (fumarato-redutase solúvel, `fum_m + q6h2_m → succ_m + q6_m`)
é preenchimento de lacuna guiado por fenótipo: sem ela a *T. delbrueckii* e a *L. thermotolerans* dão
µ = 0 em anaerobiose e **não há consórcio anaeróbio nenhum para simular**. A célula abaixo mede as duas
situações em vez de as afirmar.

A *M. pulcherrima* não tem variante anaeróbia (decisão registada no `obj1b`, § 11.2). Corre em
microaerobiose declarada, q(O₂) = 0.5 mmol/gDW/h.

In [7]:
O2_RID = {cod: next(r.id for r in m.exchanges
                    if MM._norm(next(iter(r.metabolites)).name) in ('oxygen', 'dioxygen'))
          for cod, m in M.items()}
BIO = {'Sc': 'r_2111', 'Td': 'BIOMASS', 'Mp': 'BIOMASS', 'Lt': 'BIOMASS'}

print('Efeito da NS1, medido (MS300 + suplementos, O2 = 0):')
for cod in ('Td', 'Lt'):
    for ids in (None, ['NS1', 'NS2b', 'NS3']):
        m = cobra.io.read_sbml_model(CURADOS[cod])
        o2 = next(r.id for r in m.exchanges
                  if MM._norm(next(iter(r.metabolites)).name) in ('oxygen', 'dioxygen'))
        CU.modo_anaerobio(m, cod, o2_rid=o2, ids=ids, verbose=False)
        MM.aplicar_meio_mosto(m, q_o2=0.0, suplementos=MM.SUPLEMENTOS, verbose=False)
        m.objective = BIO[cod]
        etiqueta = 'sem NS1 (por omissao)' if ids is None else 'com NS1'
        print(f'   {cod}  {etiqueta:22s} mu = {m.slim_optimize():.4f}')

Efeito da NS1, medido (MS300 + suplementos, O2 = 0):
   Td  sem NS1 (por omissao)  mu = 0.0000
   Td  com NS1                mu = 0.1835
   Lt  sem NS1 (por omissao)  mu = 0.0000
   Lt  com NS1                mu = 0.1835


### 3.2 · Aplicar o regime e exportar

In [8]:
ANAEROBIAS = {'Sc': ['SC1'], 'Td': ['NS1', 'NS2b', 'NS3'], 'Lt': ['NS1', 'NS2b', 'NS3']}

for cod in ('Sc', 'Td', 'Lt'):
    log = CU.modo_anaerobio(M[cod], cod, o2_rid=O2_RID[cod], ids=ANAEROBIAS[cod], verbose=False)
    print(f'{cod}: anaerobiose, edicoes {ANAEROBIAS[cod]} ({len(log)} operacoes)')
log = CU.modo_microaerobio(M['Mp'], 'Mp', o2_rid=O2_RID['Mp'], regime='microaerobiose')
print(f"Mp: microaerobiose, q(O2) = {CM.Q_O2['Mp']} mmol/gDW/h")

FICH_HARM = {}
for cod, m in M.items():
    m.id = CM.CODIGOS[cod]
    p = os.path.join(SAIDA, f'{CM.CODIGOS[cod]}_harm.xml')
    cobra.io.write_sbml_model(m, p)
    FICH_HARM[cod] = p
print('\nExportados (harmonizado + regime):')
for cod, p in FICH_HARM.items():
    print(f'   {cod}: {os.path.basename(p)}')

Sc: anaerobiose, edicoes ['SC1'] (3 operacoes)
Td: anaerobiose, edicoes ['NS1', 'NS2b', 'NS3'] (7 operacoes)
Lt: anaerobiose, edicoes ['NS1', 'NS2b', 'NS3'] (7 operacoes)
Mp: microaerobiose, q(O2) = 0.5 mmol/gDW/h

Exportados (harmonizado + regime):
   Sc: Scer_harm.xml
   Lt: Ltho_harm.xml
   Mp: Mpul_harm.xml
   Td: Tdel_harm.xml


### 3.3 · O meio muda de lugar quando se passa de modelo isolado para comunidade

Num modelo isolado, a única fonte é o meio: o `meio_mosto.aplicar_meio_mosto` fecha **toda** a captação e
depois abre a receita. Está certo, e é o que produziu os µ do `obj2`.

Numa comunidade isso deixa de estar certo. Quem define o meio passa a ser o pool partilhado
(`Environment.apply(exclusive=True)`), e um composto que não está na receita **só pode entrar num pool se um
parceiro o tiver secretado**. Manter as trocas do organismo fechadas nesse cenário não restringe o meio:
bloqueia exactamente a alimentação cruzada que a análise existe para medir.

Medido: com as trocas de cada organismo fechadas fora da receita, a *S. cerevisiae* num par com a
*T. delbrueckii* dá µ = 0.1654, igual ao solo. Com elas abertas, dá 0.1704, e a diferença vem de captações
de `man6p`, `uri`, `thymd` e `oaa` que só existem porque a parceira as secreta. Fechá-las tornaria todos os
índices de alimentação cruzada estruturalmente nulos.

Por isso os modelos entram na comunidade com a captação aberta em todas as trocas e só o **oxigénio** no
valor do regime. O oxigénio tem de estar no modelo-fonte e não no modelo fundido: o `mip_score` reconstrói
a comunidade a partir dos modelos-fonte (`community.copy(copy_models=False)`), e um bound escrito só no
fundido desaparece nessa cópia.

In [9]:
FICH_SM = {}
for cod, p in FICH_HARM.items():
    m = cobra.io.read_sbml_model(p)
    info = CM.preparar_para_comunidade(m, cod)
    q = os.path.join(SAIDA, f'{CM.CODIGOS[cod]}_smetana.xml')
    cobra.io.write_sbml_model(m, q)
    FICH_SM[cod] = q
    print(f"{cod}: {info['trocas_abertas']} trocas abertas, O2 ({info['o2']}) a {info['q_o2']}")

Sc: 269 trocas abertas, O2 (EX_o2_e) a -0.0
Lt: 251 trocas abertas, O2 (EX_o2_e) a -0.0
Mp: 253 trocas abertas, O2 (EX_o2_e) a -0.5
Td: 239 trocas abertas, O2 (EX_o2_e) a -0.0


### 3.4 · Teste de aceitação: a harmonização não pode ter mudado nada

O critério é simples e é verificável: aplicar o MS300 a cada modelo **harmonizado**, isolado, no seu próprio
regime, tem de devolver exactamente o µ medido no `obj2`. Se devolver outra coisa, a renomeação alterou o
modelo e tudo o que vem a seguir é sobre modelos diferentes dos que foram curados.

In [10]:
linhas = []
for cod, p in FICH_HARM.items():
    m = cobra.io.read_sbml_model(p)
    MM.aplicar_meio_mosto(m, q_o2=CM.Q_O2[cod], suplementos=MM.SUPLEMENTOS, verbose=False)
    m.objective = BIO[cod]
    mu = m.slim_optimize()
    linhas.append({'cod': cod, 'especie': CM.ESPECIES[cod], 'regime': f'q(O2)={CM.Q_O2[cod]}',
                   'mu_harmonizado': round(mu, 4), 'mu_obj2': MU_OBJ2[cod],
                   'delta': round(mu - MU_OBJ2[cod], 6)})
aceit = pd.DataFrame(linhas)
display(aceit)

if aceit.delta.abs().max() > 1e-3:
    raise RuntimeError('A harmonizacao alterou o comportamento dos modelos. Parar aqui.')
print('Harmonizacao neutra: os quatro modelos reproduzem o mu do obj2.')

,cod,especie,regime,mu_harmonizado,mu_obj2,delta
0,Sc,Saccharomyces cerevisiae,q(O2)=0.0,0.1654,0.1654,0.000024
1,Lt,Lachancea thermotolerans,q(O2)=0.0,0.1835,0.1835,0.000033
2,Mp,Metschnikowia pulcherrima,q(O2)=0.5,0.1836,0.1836,0.000000
3,Td,Torulaspora delbrueckii,q(O2)=0.0,0.1835,0.1835,0.000033


Harmonizacao neutra: os quatro modelos reproduzem o mu do obj2.


## 4 · Construção das comunidades

Sete consórcios: os seis pares e o quarteto.

O par é a unidade de decisão. É um par que se inocula num ensaio de vinificação, em co-inoculação ou em
inoculação sequencial, e é sobre pares que os índices do SMETANA têm interpretação directa — o MRO e o SC
são definidos entre dois organismos e, com mais membros, viram médias sobre pares. O quarteto entra para
medir se a sobreposição de nichos cresce com o número de membros, não como proposta de ensaio.

`merge_extracellular_compartments=False` é obrigatório: cada organismo tem de manter as suas próprias
trocas no modelo fundido, senão o MIP e o MRO não têm como distinguir quem capta o quê.

In [11]:
Rf = {cod: load_cbmodel(p, flavor='bigg') for cod, p in FICH_SM.items()}
INV = {v: k for k, v in CM.CODIGOS.items()}
DESENHOS = CM.desenhos()
print('Consorcios:', list(DESENHOS))

def construir(cods, cid=None):
    com = Community(cid or '_'.join(cods), models=[Rf[c] for c in cods], copy_models=True,
                    merge_extracellular_compartments=False, create_biomass=True, interacting=True)
    env, rel = CM.ambiente_mosto(com, mapa_met, codigos=cods)
    return com, env, rel, {o: INV[o] for o in com.organisms}

com0, env0, rel0, orgs0 = construir(['Sc', 'Td'])
print(f'\nSc+Td: {len(com0.merged.reactions)} reacoes, {len(com0.merged.metabolites)} metabolitos')
print(f"pools: {len([r for r in com0.merged.reactions if r.endswith('_pool')])}")
print(f"pools abertos pelo MS300: {rel0['n_pool']}")
print(f"compostos da receita sem pool: {[f[0] for f in rel0['fora']]}")

Consorcios: ['Sc+Td', 'Sc+Mp', 'Sc+Lt', 'Td+Mp', 'Td+Lt', 'Mp+Lt', 'Sc+Td+Mp+Lt']

Sc+Td: 6437 reacoes, 4621 metabolitos
pools: 361
pools abertos pelo MS300: 47
compostos da receita sem pool: ['iodide', 'cobalt', 'borate', 'molybdate']


## 5 · Validação estrutural

Antes de calcular índice nenhum, confirmar que a estrutura é a que se julga ter.

In [12]:
mg0 = com0.merged
pools = {r[7:-5] if r.startswith('R_EX_M_') else r[5:-5]
         for r in mg0.reactions if r.endswith('_pool')}

testes = [
    ('Comunidade com 2 organismos',            len(com0.organisms) == 2),
    ('R_Community_Growth presente',            'R_Community_Growth' in mg0.reactions),
    ('Sink_biomass de cada organismo',         all(f'Sink_biomass_{o}' in mg0.reactions for o in com0.organisms)),
    ('Trocas proprias por organismo',          all(len(com0.organisms_exchange_reactions.get(o, ())) > 100
                                                   for o in com0.organisms)),
    ('Pools criados',                          len(pools) > 0),
    ('Sem metabolito extracelular sem _e',     all(m.id.endswith('_e') for m in M['Sc'].metabolites
                                                   if m.compartment == 'e')),
    ('Glucose com pool comum',                 CM._pool(mapa_met['glucose']) in mg0.reactions),
    ('Azoto amoniacal com pool comum',         CM._pool(mapa_met['ammonium']) in mg0.reactions),
    ('Pools do MS300 abertos >= 40',           rel0['n_pool'] >= 40),
]
falhas = [d for d, ok in testes if not ok]
for d, ok in testes:
    print(f"  [{'v' if ok else 'X'}]  {d}")
if falhas:
    raise RuntimeError(f'Validacao falhou: {falhas}')
print('\nEstrutura validada.')

  [v]  Comunidade com 2 organismos
  [v]  R_Community_Growth presente
  [v]  Sink_biomass de cada organismo
  [v]  Trocas proprias por organismo
  [v]  Pools criados
  [v]  Sem metabolito extracelular sem _e
  [v]  Glucose com pool comum
  [v]  Azoto amoniacal com pool comum
  [v]  Pools do MS300 abertos >= 40

Estrutura validada.


## 6 · Crescimento por consórcio

Três leituras por organismo, e a diferença entre elas é o que interessa:

| leitura | o que mede |
|---|---|
| **solo** | o modelo sozinho, com o MS300 aplicado à maneira do `obj2` |
| **no consórcio, sozinho a crescer** | o organismo no modelo fundido, com o parceiro presente mas com a biomassa dele não exigida |
| **comunidade** | `R_Community_Growth`, que obriga todos os membros a crescer em proporção fixa |

A segunda leitura é a que revela alimentação cruzada: se subir acima do solo, o organismo está a usar
algo que o parceiro secreta.

In [13]:
def crescimentos(cods):
    com, env, rel, orgs = construir(cods)
    env.apply(com.merged, exclusive=True, inplace=True)
    out = {}
    for org, bio in com.organisms_biomass_reactions.items():
        s = FBA(com.merged, objective=bio)
        out[orgs[org]] = round(s.fobj or 0.0, 4)
    s = FBA(com.merged, objective='R_Community_Growth')
    return out, round(s.fobj or 0.0, 4), com, env, orgs

t0 = time.time()
CRESC, COMUNS = {}, {}
for nome, cods in DESENHOS.items():
    indiv, comunidade, com, env, orgs = crescimentos(cods)
    CRESC[nome] = {'membros': cods, 'individual': indiv, 'comunidade': comunidade}
    COMUNS[nome] = (cods, orgs)
    print(f'{nome:16s} {indiv}  comunidade={comunidade}')
print(f'[{time.time()-t0:.0f}s]')

linhas = []
for nome, d in CRESC.items():
    for cod in d['membros']:
        linhas.append({'consorcio': nome, 'cod': cod,
                       'mu_solo': MU_OBJ2[cod],
                       'mu_no_consorcio': d['individual'][cod],
                       'ganho': round(d['individual'][cod] - MU_OBJ2[cod], 4),
                       'mu_comunidade': d['comunidade']})
tab_cresc = pd.DataFrame(linhas)
display(tab_cresc)

Sc+Td            {'Sc': 0.1704, 'Td': 0.2}  comunidade=0.092
Sc+Mp            {'Sc': 0.1704, 'Mp': 0.2}  comunidade=0.092
Sc+Lt            {'Sc': 0.1704, 'Lt': 0.2}  comunidade=0.092
Td+Mp            {'Td': 0.1836, 'Mp': 0.1836}  comunidade=0.0918
Td+Lt            {'Td': 0.1835, 'Lt': 0.1835}  comunidade=0.0918
Mp+Lt            {'Mp': 0.1836, 'Lt': 0.1836}  comunidade=0.0918
Sc+Td+Mp+Lt      {'Sc': 0.1704, 'Td': 0.2, 'Mp': 0.2, 'Lt': 0.2}  comunidade=0.0479
[5s]


,consorcio,cod,mu_solo,mu_no_consorcio,ganho,mu_comunidade
0,Sc+Td,Sc,0.1654,0.1704,0.0050,0.0920
1,Sc+Td,Td,0.1835,0.2000,0.0165,0.0920
2,Sc+Mp,Sc,0.1654,0.1704,0.0050,0.0920
3,Sc+Mp,Mp,0.1836,0.2000,0.0164,0.0920
4,Sc+Lt,Sc,0.1654,0.1704,0.0050,0.0920
5,Sc+Lt,Lt,0.1835,0.2000,0.0165,0.0920
6,Td+Mp,Td,0.1835,0.1836,0.0001,0.0918
7,Td+Mp,Mp,0.1836,0.1836,0.0000,0.0918
8,Td+Lt,Td,0.1835,0.1835,0.0000,0.0918
9,Td+Lt,Lt,0.1835,0.1835,0.0000,0.0918


## 7 · `min_growth`: um parâmetro que não se escolhe por conveniência

Os cinco índices do SMETANA recebem um `min_growth`, o crescimento mínimo imposto ao resolver os problemas
de meio mínimo. O notebook de referência das leguminosas usava `0.1 × min(µ)`. Aqui isso dá 0.017 e o MIP
falha; a 0.001 o MIP corre e é o MRO que falha. Não há um valor que sirva os dois.

Escolher o número que faz o código correr é ajuste de parâmetro, e é a única coisa que esta análise não
pode fazer. A alternativa é medir: correr cada índice ao longo de uma gama de `min_growth` e reportar em
que gama cada um é viável e se o valor muda lá dentro.

Duas coisas que a varredura torna explícitas e que mudam a leitura de tudo o que vem a seguir:

1. **Os índices do SMETANA não leem os valores dos bounds.** Leem o ambiente como a *lista* de compostos
   disponíveis e reabrem cada um ao seu próprio `max_uptake` para procurar meios mínimos. O MS300 entra
   aqui como composição qualitativa, não como taxas. Os bounds da equação de Sauer do `obj2` governam as
   leituras de FBA da secção 6 e **não** governam MIP, MRO, SC, MUS e MPS. Foi por isso que, na primeira
   versão, o oxigénio aparecia no meio mínimo de um consórcio anaeróbio: estava no ambiente com bound zero,
   e um bound zero não fecha nada — só a ausência do ambiente fecha.
2. **O MIP não é determinista.** É um MILP com óptimos alternativos: a mesma chamada deu 10 e 4 em duas
   execuções com `min_growth` diferentes por um factor de dez. Um MIP reportado sem gama é um número sem
   incerteza declarada.

In [14]:
GAMA = [0.0001, 0.001, 0.005, 0.01, 0.017, 0.05]
INDICES_RAPIDOS = [('MIP', mip_score), ('MRO', mro_score), ('SC', sc_score)]

def valor(nome, r):
    if r is None:
        return None
    if nome in ('MIP', 'MRO'):
        return r[0]
    return max((v for d in r.values() for v in (d or {}).values()), default=None)

t0 = time.time(); varredura = []
for mgv in GAMA:
    for nome, fn in INDICES_RAPIDOS:
        com, env, _, _ = construir(['Sc', 'Td'])
        try:
            v = valor(nome, fn(com, environment=env, min_growth=mgv, verbose=False))
        except Exception as e:
            v = f'ERRO {type(e).__name__}'
        varredura.append({'min_growth': mgv, 'indice': nome, 'valor': v,
                          'viavel': v is not None and not isinstance(v, str)})
print(f'[{time.time()-t0:.0f}s]')

vv = pd.DataFrame(varredura)
display(vv.pivot(index='min_growth', columns='indice', values='valor'))

viaveis = {n: sorted(vv[(vv.indice == n) & vv.viavel].min_growth) for n, _ in INDICES_RAPIDOS}
print('Gama viavel por indice:')
for n, g in viaveis.items():
    print(f'   {n:4s} {g}')

# Regra escrita antes de ver os resultados: para cada indice, a mediana da sua
# propria gama viavel. Nao e um valor escolhido por dar melhor numero — e o
# ponto medio do intervalo em que o indice tem solucao, e a tabela acima mostra
# se o valor do indice se mexe dentro desse intervalo.
RECURSO = 0.001
MIN_GROWTH = {n: (g[len(g)//2] if g else RECURSO) for n, g in viaveis.items()}
MIN_GROWTH['MUS'] = MIN_GROWTH['MPS'] = MIN_GROWTH['SC']
print(f'\nmin_growth adoptado por indice: {MIN_GROWTH}')
for n, g in viaveis.items():
    if not g:
        print(f'   AVISO: {n} nao foi viavel em nenhum ponto da gama; usa {RECURSO} por recurso.')

estavel = vv[vv.viavel].groupby('indice').valor.nunique()
print('\nValores distintos dentro da gama viavel (1 = indice estavel):')
print(estavel.to_string())

[21s]


indice,MIP,MRO,SC
min_growth,,,
0.0001,NaN,NaN,1.0
0.0010,NaN,0.177778,1.0
0.0050,1.0,0.347826,1.0
0.0100,1.0,0.320000,1.0
0.0170,1.0,0.320000,1.0
0.0500,NaN,0.413793,1.0


Gama viavel por indice:
   MIP  [0.005, 0.01, 0.017]
   MRO  [0.001, 0.005, 0.01, 0.017, 0.05]
   SC   [0.0001, 0.001, 0.005, 0.01, 0.017, 0.05]

min_growth adoptado por indice: {'MIP': 0.01, 'MRO': 0.01, 'SC': 0.01, 'MUS': 0.01, 'MPS': 0.01}

Valores distintos dentro da gama viavel (1 = indice estavel):
indice
MIP    1
MRO    4
SC     1


## 8 · Índices do SMETANA

| índice | o que mede | leitura |
|---|---|---|
| **MIP** | compostos que o consórcio deixa de precisar de ir buscar ao meio porque os troca entre si | quanto maior, mais complementaridade |
| **MRO** | sobreposição dos requisitos mínimos de cada membro | 0 = nichos disjuntos, 1 = competição total |
| **SC** | frequência com que um membro depende do outro para crescer | probabilidade estrutural de dependência |
| **MUS** | probabilidade de captar um composto produzido pelo parceiro | por metabolito |
| **MPS** | compostos que cada membro consegue secretar e que não vêm do meio | binário na implementação, não gradual |

O MPS é reportado como binário e não como a medida graduada descrita em Zelezniak et al. (2015). É uma
diferença entre o artigo e a implementação, confirmada no repositório do SMETANA (issue #14), e muda a
interpretação: um MPS de 1.0 quer dizer "consegue secretar", não "secreta com frequência 1".

Cada índice corre numa comunidade acabada de construir. Os índices alteram bounds do modelo fundido, e
reutilizar a mesma instância contamina o índice seguinte.

**Os índices dividem-se em dois grupos por custo, e o segundo não converge com este solver.** O MIP, o MRO
e o SC resolvem um número fixo de problemas e correm em 15 a 40 s por consórcio. O MUS e o MPS amostram
`n_solutions` soluções alternativas de um MILP e devolvem a frequência com que cada metabolito aparece.
Com o SCIP, medido no par Sc+Td:

| `n_solutions` | metabolitos com MUS > 0 | tempo |
|---|---|---|
| 3 | 19 | 109 s |
| 5 | 23 | 151 s |

O valor publicado por omissão é 100. A contagem ainda sobe entre 3 e 5, ou seja **não convergiu**, e uma
frequência calculada sobre 5 amostras só pode tomar os valores 0, 0.2, 0.4, 0.6, 0.8 e 1: a leitura
probabilística do MUS não sobrevive a esta amostragem. Por isso, neste notebook:

- MIP, MRO e SC correm nos sete consórcios;
- MUS e MPS correm só nos três pares com *S. cerevisiae*, que são os que decidem a etapa seguinte;
- o valor numérico do MUS e do MPS **não é lido como probabilidade**. Usa-se apenas a lista de metabolitos
  com score acima de zero, que é uma afirmação estrutural ("consegue", "não consegue") e não estatística;
- a secção 9.1 repete o teste de convergência para ficar no registo da execução.

Repetir com Gurobi antes de dar qualquer número de MUS ou MPS como resultado.

*Tempo de execução desta secção e da seguinte: cerca de 25 a 35 minutos.*

In [15]:
N_SOLUTIONS = 5          # ver a discussao acima: nao convergido, so a lista e usada
PARES_SC = [n for n, cods in DESENHOS.items() if 'Sc' in cods and len(cods) == 2]
print(f'MUS/MPS so em: {PARES_SC}')

def indices_do_consorcio(cods, nome, completos=True):
    res, tempos, orgs = {}, {}, None
    tarefas = [('MIP', mip_score, 'MIP'), ('MRO', mro_score, 'MRO'), ('SC', sc_score, 'SC')]
    if completos:
        tarefas += [('MUS', mu_score, 'MUS'), ('MPS', mp_score, 'MPS')]
    for etiqueta, fn, chave in tarefas:
        com, env, _, orgs = construir(cods)
        t = time.time()
        # O mp_score tem outra assinatura: mp_score(community, environment, abstol).
        # Nao recebe min_growth, n_solutions nem verbose. Passar-lhos da TypeError.
        if etiqueta == 'MPS':
            kw = dict(environment=env)
        else:
            kw = dict(environment=env, min_growth=MIN_GROWTH[chave], verbose=False)
            if etiqueta == 'MUS':
                kw['n_solutions'] = N_SOLUTIONS
        try:
            res[etiqueta] = fn(com, **kw)
        except Exception as e:
            res[etiqueta] = None
            print(f'   {nome} {etiqueta}: ERRO {type(e).__name__}')
        tempos[etiqueta] = round(time.time() - t, 1)
    for f in ('MUS', 'MPS'):
        res.setdefault(f, None)
    res['_orgs'] = orgs
    res['_tempos'] = tempos
    return res

t0 = time.time(); RES = {}
for nome, cods in DESENHOS.items():
    print(f'{nome} ...', end=' ', flush=True)
    RES[nome] = indices_do_consorcio(cods, nome, completos=(nome in PARES_SC))
    print(RES[nome]['_tempos'], flush=True)
print(f'total [{time.time()-t0:.0f}s]')

MUS/MPS so em: ['Sc+Td', 'Sc+Mp', 'Sc+Lt']
Sc+Td ... {'MIP': 0.4, 'MRO': 0.8, 'SC': 0.6, 'MUS': 3.5, 'MPS': 5.6}
Sc+Mp ... {'MIP': 0.8, 'MRO': 1.1, 'SC': 0.4, 'MUS': 2.2, 'MPS': 4.0}
Sc+Lt ... {'MIP': 0.5, 'MRO': 1.0, 'SC': 0.6, 'MUS': 4.1, 'MPS': 5.1}
Td+Mp ... {'MIP': 0.4, 'MRO': 0.7, 'SC': 0.3}
Td+Lt ... {'MIP': 0.5, 'MRO': 1.8, 'SC': 0.4}
Mp+Lt ... {'MIP': 0.4, 'MRO': 0.2, 'SC': 0.5}
Sc+Td+Mp+Lt ... {'MIP': 1.4, 'MRO': 5.0, 'SC': 1.7}
total [57s]


In [16]:
def _limpa_id(m):
    """`M_pntoR_e` -> `pntoR`. Os indices devolvem o id do metabolito no modelo
    fundido, com prefixo `M_` e sufixo de compartimento."""
    m = m[2:] if m.startswith('M_') else m
    return m[:-2] if m.endswith('_e') else m

def resumo(nome, r):
    mip = r['MIP'][0] if r['MIP'] else None
    mro = r['MRO'][0] if r['MRO'] else None
    sc_max = max((v for d in (r['SC'] or {}).values() for v in (d or {}).values()), default=None)
    n_mus = sum(1 for d in (r['MUS'] or {}).values() for v in (d or {}).values() if v and v > 0)
    n_mps = sum(1 for d in (r['MPS'] or {}).values() for v in (d or {}).values() if v and v > 0)
    return {'consorcio': nome, 'n': len(DESENHOS[nome]),
            'MIP': mip, 'MRO': None if mro is None else round(mro, 4),
            'SC_max': sc_max, 'MUS_n': n_mus, 'MPS_n': n_mps}

tab_ind = pd.DataFrame([resumo(n, r) for n, r in RES.items()])
display(tab_ind.sort_values(['n', 'MRO']))
print(f"min_growth usado: {MIN_GROWTH}")

,consorcio,n,MIP,MRO,SC_max,MUS_n,MPS_n
1,Sc+Mp,2,NaN,0.1111,1.0,23,216
4,Td+Lt,2,0.0,0.3077,0.0,0,0
0,Sc+Td,2,1.0,0.3200,1.0,25,177
2,Sc+Lt,2,1.0,0.4167,1.0,32,179
3,Td+Mp,2,1.0,NaN,0.0,0,0
5,Mp+Lt,2,1.0,NaN,0.0,0,0
6,Sc+Td+Mp+Lt,4,NaN,0.3333,1.0,0,0


min_growth usado: {'MIP': 0.01, 'MRO': 0.01, 'SC': 0.01, 'MUS': 0.01, 'MPS': 0.01}


### 8.1 · O MRO dos pares com *S. cerevisiae* está deflacionado por construção

O MRO mede a sobreposição dos meios mínimos individuais. Oito compostos **não podem sobrepor-se por razão
estrutural e não biológica**: os sete minerais que só o yeast-GEM tem como trocas separadas (K, Na, Ca, Mn,
Cu, Zn, Mg) e o `ash 1 g` que só os CarveFungi têm. Num par Sc + não-*Saccharomyces*, esses oito entram no
denominador e nunca no numerador.

O efeito é sistemático e só numa direcção: baixa o MRO de todos os pares com *S. cerevisiae* e não toca nos
pares entre não-*Saccharomyces*. Ler a tabela acima sem isto levaria à conclusão errada de que a
*S. cerevisiae* compete menos com as não-*Saccharomyces* do que elas entre si.

A correcção recalcula o MRO a partir dos meios individuais que o `mro_score` devolve, retirando esses oito
compostos dos dois lados. Não é uma estimativa: é a mesma fórmula sobre o mesmo resultado, com o eixo que
sabemos não ser comparável posto de fora.

In [17]:
NAO_COMPARAVEIS = {'ash_1g'} | {f's_{n}' for n in
                   ('1374', '1438', '4199', '4200', '4201', '4202', '4203', '4204')}

def mro_corrigido(extras, excluir=NAO_COMPARAVEIS):
    if not extras:
        return None, None
    ind = {o: {_limpa_id(m) for m in mets} - excluir
           for o, mets in extras['individual_media'].items()}
    pares = [(a, b) for a, b in itertools.combinations(ind, 2)]
    if not pares:
        return None, None
    num = sum(len(ind[a] & ind[b]) for a, b in pares) / len(pares)
    den = sum(map(len, ind.values())) / len(ind)
    return (num / den if den else None), {o: len(v) for o, v in ind.items()}

linhas = []
for nome, r in RES.items():
    bruto = r['MRO'][0] if r['MRO'] else None
    corr, tam = mro_corrigido(r['MRO'][1] if r['MRO'] else None)
    linhas.append({'consorcio': nome, 'tem_Sc': 'Sc' in DESENHOS[nome],
                   'MRO_bruto': None if bruto is None else round(bruto, 4),
                   'MRO_corrigido': None if corr is None else round(corr, 4),
                   'delta': None if (bruto is None or corr is None) else round(corr - bruto, 4),
                   'meio_individual': tam})
tab_mro = pd.DataFrame(linhas)
display(tab_mro)

com_sc = tab_mro[tab_mro.tem_Sc & tab_mro.delta.notna()].delta
sem_sc = tab_mro[~tab_mro.tem_Sc & tab_mro.delta.notna()].delta
print(f'Efeito medio da correccao, pares com Sc : {com_sc.mean() if len(com_sc) else float("nan"):+.4f}')
print(f'Efeito medio da correccao, pares sem Sc : {sem_sc.mean() if len(sem_sc) else float("nan"):+.4f}')

,consorcio,tem_Sc,MRO_bruto,MRO_corrigido,delta,meio_individual
0,Sc+Td,True,0.3200,0.5000,0.1800,"{'Scer': 9, 'Tdel': 7}"
1,Sc+Mp,True,0.1111,0.2222,0.1111,"{'Scer': 4, 'Mpul': 5}"
2,Sc+Lt,True,0.4167,0.6667,0.2500,"{'Scer': 9, 'Ltho': 6}"
3,Td+Mp,False,NaN,NaN,NaN,None
4,Td+Lt,False,0.3077,0.1818,-0.1259,"{'Tdel': 5, 'Ltho': 6}"
5,Mp+Lt,False,NaN,NaN,NaN,None
6,Sc+Td+Mp+Lt,True,0.3333,0.4444,0.1111,"{'Scer': 4, 'Tdel': 5, 'Mpul': 3, 'Ltho': 3}"


Efeito medio da correccao, pares com Sc : +0.1631
Efeito medio da correccao, pares sem Sc : -0.1259


### 8.2 · SC: o que o índice diz e o que a FBA confirma

O SC = 1.0 lê-se facilmente como dependência absoluta, e quase sempre não é isso. O `sc_score` testa
viabilidade a um `min_growth` baixo, por uma procura MILP amostrada de dadores, e não verifica se a espécie
alcança a sua taxa de crescimento real sem o parceiro.

A verificação directa: forçar a zero todas as reações do parceiro e voltar a medir. Se o µ não descer, o
SC = 1.0 é um artefacto de construção do índice e não uma auxotrofia.

In [18]:
def verificar_sc(cods):
    com, env, _, orgs = construir(cods)
    env.apply(com.merged, exclusive=True, inplace=True)
    base = {orgs[o]: FBA(com.merged, objective=b).fobj or 0.0
            for o, b in com.organisms_biomass_reactions.items()}
    linhas = []
    for alvo_org, alvo_cod in orgs.items():
        for outro_org, outro_cod in orgs.items():
            if outro_org == alvo_org:
                continue
            com2, env2, _, orgs2 = construir(cods)
            env2.apply(com2.merged, exclusive=True, inplace=True)
            bio_outro = com2.organisms_biomass_reactions[outro_org]
            for rid in com2.organisms_reactions[outro_org]:
                if rid != bio_outro:
                    com2.merged.set_flux_bounds(rid, 0, 0)
            s = FBA(com2.merged, objective=com2.organisms_biomass_reactions[alvo_org])
            linhas.append({'consorcio': '+'.join(cods), 'organismo': alvo_cod,
                           'parceiro_desligado': outro_cod,
                           'mu_com_parceiro': round(base[alvo_cod], 5),
                           'mu_sem_parceiro': round(s.fobj or 0.0, 5),
                           'depende': (s.fobj or 0.0) < 0.9 * base[alvo_cod]})
    return linhas

t0 = time.time()
ver = [l for cods in DESENHOS.values() if len(cods) == 2 for l in verificar_sc(cods)]
tab_ver = pd.DataFrame(ver)
display(tab_ver)
print(f'[{time.time()-t0:.0f}s]  Dependencias confirmadas por FBA: {tab_ver.depende.sum()} de {len(tab_ver)}')

,consorcio,organismo,parceiro_desligado,mu_com_parceiro,mu_sem_parceiro,depende
0,Sc+Td,Sc,Td,0.17042,0.16542,False
1,Sc+Td,Td,Sc,0.20002,0.18353,False
2,Sc+Mp,Sc,Mp,0.17042,0.17042,False
3,Sc+Mp,Mp,Sc,0.19999,0.18360,False
4,Sc+Lt,Sc,Lt,0.17042,0.16542,False
5,Sc+Lt,Lt,Sc,0.20002,0.18353,False
6,Td+Mp,Td,Mp,0.18363,0.18363,False
7,Td+Mp,Mp,Td,0.18360,0.18360,False
8,Td+Lt,Td,Lt,0.18353,0.18353,False
9,Td+Lt,Lt,Td,0.18353,0.18353,False


[8s]  Dependencias confirmadas por FBA: 0 de 12


### 9.1 · Convergência do MUS

Se a contagem de metabolitos ainda subir com `n_solutions`, o índice não convergiu e o valor numérico não é
uma probabilidade. O teste fica no notebook para que a afirmação seja verificável e não apenas declarada
nas limitações.

In [19]:
conv = []
for n in (3, 5):
    com, env, _, _ = construir(['Sc', 'Td'])
    t = time.time()
    r = mu_score(com, environment=env, min_growth=MIN_GROWTH['MUS'], n_solutions=n, verbose=False)
    k = sum(1 for d in (r or {}).values() for v in (d or {}).values() if v and v > 0)
    conv.append({'n_solutions': n, 'metabolitos_MUS_acima_de_zero': k, 'segundos': round(time.time() - t, 1)})
tab_conv = pd.DataFrame(conv)
display(tab_conv)
if tab_conv['metabolitos_MUS_acima_de_zero'].nunique() > 1:
    print('Nao convergido: a contagem muda com n_solutions. Nao ler o MUS como probabilidade.')

,n_solutions,metabolitos_MUS_acima_de_zero,segundos
0,3,25,2.1
1,5,25,3.4


## 9 · Alimentação cruzada

O MUS e o MPS em bruto incluem ruído que não é interação célula a célula: minerais do meio partilhado,
protões, água, CO₂, e o pseudo-metabolito `ash 1 g`. São filtrados antes de categorizar.

As categorias são as do problema, não as do notebook das leguminosas: o azoto assimilável (YAN), que é o
que limita a fermentação vínica; os ácidos orgânicos, onde está o ácido L-láctico da
*L. thermotolerans* que é o mecanismo de acidificação de interesse; os esteróis e ácidos gordos
insaturados, que são a limitação declarada do `obj2`; e os compostos de Ehrlich, cuja **ausência** dos
modelos CarveFungi é ela própria um resultado a declarar.

In [20]:
# Os indices devolvem `M_<id>_e`. Sem normalizar, nenhum filtro e nenhuma
# categoria acertam, e tudo cai em 'outros'.
ARTEFACTOS = {'h', 'h2o', 'co2', 'o2', 'pi', 'so4', 'so3', 'nh4', 'ash_1g',
              'na1', 'k', 'cl', 'ca2', 'mg2', 'mn2', 'zn2', 'cu2', 'fe2', 'fe3'}
ARTEFACTOS |= {f's_{n}' for n in ('1374', '1438', '4199', '4200', '4201', '4202', '4203', '4204')}

# O que o MS300 fornece. Captar isto nao e alimentacao cruzada: e usar o meio.
DO_MEIO = {_limpa_id(m) for m in mapa_met.values() if m}

CATEGORIAS = {
    'azoto_YAN': {'alaL','argL','asnL','aspL','cysL','glnL','gluL','gly','hisL','ileL','leuL',
                  'lysL','metL','pheL','proL','serL','thrL','trpL','tyrL','valL','orn','homL'},
    'vitaminas': {'thm','ribflv','pntoR','nac','btn','fol','pydxn','pydx','inost','nmn','4abz','fmn'},
    'acidos_organicos': {'lacL','lacD','ac','succ','malL','cit','pyr','for','akg','oaa','fum',
                         'glx','but','oca','dca','hxa'},
    'acucares': {'glcD','fru','man','gal','tre','sucr','xylD','ribD','sbtD','mnl','glcn','gam',
                 'abtD','srbL','g1p','g6p','gam6p','man6p','f6p'},
    'fermentacao': {'etoh','glyc','actnR','btdRR','acald','dha'},
    'lipidos_esteroides': {'ergst','9Zocda','9Zhxda','ocda','hdca','lanost','zymst','9Z12Zlnlc'},
    'nucleotidos': {'ade','gua','ura','ins','adn','gsn','uri','xan','csn','din','hxan','cmp','gmp',
                    'ump','amp','thymd','dtmp','dttp','pap','paps'},
    'redox_tiois': {'gthrd','gthox','cys','met'},
    # Identificadores confirmados nos proprios ficheiros, nao adivinhados.
    # Reconciliados (existem nos dois lados): 2mbald, 2mbtoh, pacald.
    # So nos CarveFungi: 2mppal, pac, peamn, ind3ac, iad, pad, 3mop, 4hoxpac.
    # So no yeast-GEM, sem par: s_0186 (2-feniletanol), s_0235 (3-metilbutanal),
    # s_1530 (triptofol), s_4249 (tirosol), s_1317 (acetato de fenetilo).
    'aroma_ehrlich': {'2mbald','2mbtoh','2mppal','pacald','pac','peamn','ind3ac','iad','pad',
                      '3mop','3mob','4hoxpac','4hoxpacd','2hyoxplac','mtpp','3mppal','2mbac',
                      'iamac','ibutac','4hpheetoh',
                      's_0186','s_0235','s_1530','s_4249','s_1317'},
}

def categoria(met):
    for cat, membros in CATEGORIAS.items():
        if met in membros:
            return cat
    return 'outros'

def estatuto(met):
    if met in ARTEFACTOS:
        return 'artefacto'
    if met in DO_MEIO:
        return 'do_meio'
    return 'troca'

linhas = []
for nome, r in RES.items():
    orgs = r['_orgs'] or {}
    for etiqueta in ('MUS', 'MPS'):
        for org, mets in (r[etiqueta] or {}).items():
            for m, v in (mets or {}).items():
                if not v or v <= 0:
                    continue
                mid = _limpa_id(m)
                linhas.append({'consorcio': nome, 'cod': orgs.get(org, org), 'indice': etiqueta,
                               'metabolito': mid, 'estatuto': estatuto(mid),
                               'categoria': categoria(mid), 'score': round(v, 3)})
cruz = pd.DataFrame(linhas)
print(f'Registos: {len(cruz)}')
if len(cruz):
    print('\nPor estatuto (so `troca` e alimentacao cruzada):')
    display(pd.crosstab(cruz.estatuto, cruz.indice))
    troca = cruz[cruz.estatuto == 'troca']
    print(f'\nAlimentacao cruzada verdadeira: {len(troca)} registos')
    if len(troca):
        display(pd.crosstab(troca.categoria, [troca.indice, troca.cod]))

Registos: 652

Por estatuto (so `troca` e alimentacao cruzada):


indice,MPS,MUS
estatuto,,
artefacto,5,36
do_meio,5,24
troca,562,20



Alimentacao cruzada verdadeira: 582 registos


indice             MPS              MUS         
cod                 Lt  Mp   Sc  Td  Lt Mp Sc Td
categoria                                       
acidos_organicos    13  12   25  13   0  0  0  0
acucares             7   7   12   8   0  0  5  0
aroma_ehrlich       10   8   43   4   0  0  0  0
azoto_YAN            3   4    3   4   0  0  0  0
fermentacao          5   5    9   5   0  0  0  0
lipidos_esteroides   1   2    3   1   0  0  0  0
nucleotidos         12  11   18  10   0  1  5  0
outros              52  68  111  55   1  0  1  0
redox_tiois          1   2    0   1   0  0  1  0
vitaminas            4   3    3   4   2  1  2  1

In [21]:
if len(cruz):
    print('MUS — so os compostos que NAO vem do meio nem sao artefacto\n')
    for nome in DESENHOS:
        sub = cruz[(cruz.consorcio == nome) & (cruz.indice == 'MUS') & (cruz.estatuto == 'troca')]
        if sub.empty:
            print(f'{nome}: nenhum'); continue
        print(f'{nome}:')
        for cod, g in sub.groupby('cod'):
            itens = ', '.join(f'{r.metabolito}({r.score}, {r.categoria})'
                              for r in g.sort_values('score', ascending=False).itertuples())
            print(f'   {cod} capta: {itens}')
        print()

    print('\nO que o filtro tirou, e porque (par Sc+Td, como exemplo):')
    ex = cruz[(cruz.consorcio == 'Sc+Td') & (cruz.indice == 'MUS') & (cruz.estatuto != 'troca')]
    for est, g in ex.groupby('estatuto'):
        print(f'   [{est}] {", ".join(sorted(g.metabolito.unique()))}')

MUS — so os compostos que NAO vem do meio nem sao artefacto

Sc+Td:
   Sc capta: ribD(1.0, acucares), nmn(1.0, vitaminas), cmp(1.0, nucleotidos)
   Td capta: nmn(0.2, vitaminas)

Sc+Mp:
   Mp capta: pap(1.0, nucleotidos), 4abz(0.2, vitaminas)
   Sc capta: gthrd(1.0, redox_tiois), g3pc(0.4, outros), man6p(0.4, acucares), g6p(0.2, acucares)

Sc+Lt:
   Lt capta: 4abz(0.8, vitaminas), rnam(0.8, outros), fol(0.2, vitaminas)
   Sc capta: nmn(0.8, vitaminas), pap(0.4, nucleotidos), gam6p(0.4, acucares), ins(0.4, nucleotidos), tre(0.2, acucares), uri(0.2, nucleotidos), cmp(0.2, nucleotidos)

Td+Mp: nenhum
Td+Lt: nenhum
Mp+Lt: nenhum
Sc+Td+Mp+Lt: nenhum

O que o filtro tirou, e porque (par Sc+Td, como exemplo):
   [artefacto] ash_1g, nh4, s_1374, s_1438, s_4199, s_4200, s_4201, s_4202, s_4203, s_4204, so3, so4
   [do_meio] 9Zhxda, 9Zocda, btn, ergst, fru, thm


In [22]:
if len(cruz):
    print('MPS — capacidade de secrecao fora do meio\n')
    for nome in DESENHOS:
        sub = cruz[(cruz.consorcio == nome) & (cruz.indice == 'MPS') & (cruz.estatuto == 'troca')]
        if sub.empty:
            print(f'{nome}: nenhum'); continue
        print(f'{nome}:')
        for cod, g in sub.groupby('cod'):
            cats = g.groupby('categoria').metabolito.count().to_dict()
            print(f'   {cod} (produtor): {cats}')
        print()

    print('\nCompostos de aroma de Ehrlich detetados em qualquer indice:')
    ehr = cruz[cruz.categoria == 'aroma_ehrlich']
    print('   nenhum — consistente com a ausencia destas trocas nos tres CarveFungi'
          if ehr.empty else ehr.to_string(index=False))

MPS — capacidade de secrecao fora do meio

Sc+Td:
   Sc (produtor): {'acidos_organicos': 7, 'acucares': 4, 'aroma_ehrlich': 14, 'azoto_YAN': 1, 'fermentacao': 3, 'lipidos_esteroides': 1, 'nucleotidos': 6, 'outros': 33, 'vitaminas': 1}
   Td (produtor): {'acidos_organicos': 13, 'acucares': 8, 'aroma_ehrlich': 4, 'azoto_YAN': 4, 'fermentacao': 5, 'lipidos_esteroides': 1, 'nucleotidos': 10, 'outros': 55, 'redox_tiois': 1, 'vitaminas': 4}

Sc+Mp:
   Mp (produtor): {'acidos_organicos': 12, 'acucares': 7, 'aroma_ehrlich': 8, 'azoto_YAN': 4, 'fermentacao': 5, 'lipidos_esteroides': 2, 'nucleotidos': 11, 'outros': 68, 'redox_tiois': 2, 'vitaminas': 3}
   Sc (produtor): {'acidos_organicos': 11, 'acucares': 4, 'aroma_ehrlich': 15, 'azoto_YAN': 1, 'fermentacao': 3, 'lipidos_esteroides': 1, 'nucleotidos': 6, 'outros': 45, 'vitaminas': 1}

Sc+Lt:
   Lt (produtor): {'acidos_organicos': 13, 'acucares': 7, 'aroma_ehrlich': 10, 'azoto_YAN': 3, 'fermentacao': 5, 'lipidos_esteroides': 1, 'nucleotidos': 12

### 9.2 · A via de Ehrlich: onde é que o SMETANA vê, e onde é que está cego

O anexo do `obj1b` dizia que a via de Ehrlich não está fechada nos três CarveFungi. Com a reconciliação
feita, a afirmação pode ser precisada, e a versão precisa é menos má e mais útil:

| composto | yeast-GEM | CarveFungi | pool comum? |
|---|---|---|---|
| 2-metilbutanal | `s_*` | `2mbald_e` | **sim** |
| 2-metilbutan-1-ol | `s_*` | `2mbtoh_e` | **sim** |
| fenilacetaldeído | `s_*` | `pacald_e` | **sim** |
| 2-metilpropanal | — | `2mppal_e` | não |
| fenilacetato, indol-3-acetato | — | `pac_e`, `ind3ac_e` | não |
| **2-feniletanol** | `s_0186` | — | **não** |
| **triptofol** | `s_1530` | — | **não** |
| **tirosol** | `s_4249` | — | **não** |
| **3-metilbutanal** | `s_0235` | — | **não** |
| acetato de fenetilo | `s_1317` | — | não |

O SMETANA vê a via até ao aldeído e vê um dos álcoois superiores. Fica cego exactamente nos quatro produtos
finais que têm peso sensorial em vinho — o 2-feniletanol acima de todos — porque só existem no yeast-GEM e
não têm contraparte nos ficheiros CarveFungi.

Consequência prática: uma troca de aldeídos de Ehrlich entre a *S. cerevisiae* e uma não-*Saccharomyces* é
detetável aqui; uma troca de álcoois superiores não é. Qualquer conclusão sobre aroma tem de dizer de que
lado desta linha está.

In [23]:
EHRLICH = {
    '2mbald': '2-metilbutanal', '2mbtoh': '2-metilbutan-1-ol', 'pacald': 'fenilacetaldeido',
    '2mppal': '2-metilpropanal', 'pac': 'fenilacetato', 'ind3ac': 'indol-3-acetato',
    's_0186': '2-feniletanol', 's_0235': '3-metilbutanal', 's_1530': 'triptofol',
    's_4249': 'tirosol', 's_1317': 'acetato de fenetilo',
}
linhas = []
for mid, nome in EHRLICH.items():
    presentes = [cod for cod in M if f'{mid}_e' in M[cod].metabolites]
    linhas.append({'metabolito': mid, 'composto': nome,
                   'modelos': ''.join(c if c in presentes else '·' for c in ('Sc','Td','Mp','Lt')),
                   'pool_comum': len(presentes) > 1 and 'Sc' in presentes})
tab_ehr = pd.DataFrame(linhas).sort_values('pool_comum', ascending=False)
display(tab_ehr)
print(f"Com pool comum entre Sc e as nao-Saccharomyces: {tab_ehr.pool_comum.sum()} de {len(tab_ehr)}")
print('Cegos (so no yeast-GEM):',
      ', '.join(tab_ehr[~tab_ehr.pool_comum].composto))

,metabolito,composto,modelos,pool_comum
0,2mbald,2-metilbutanal,Sc·MpLt,True
1,2mbtoh,2-metilbutan-1-ol,Sc·MpLt,True
2,pacald,fenilacetaldeido,ScTdMpLt,True
3,2mppal,2-metilpropanal,Sc·MpLt,True
4,pac,fenilacetato,·Td·Lt,False
5,ind3ac,indol-3-acetato,·TdMpLt,False
6,s_0186,2-feniletanol,Sc···,False
7,s_0235,3-metilbutanal,Sc···,False
8,s_1530,triptofol,Sc···,False
9,s_4249,tirosol,Sc···,False


Com pool comum entre Sc e as nao-Saccharomyces: 4 de 11
Cegos (so no yeast-GEM): fenilacetato, indol-3-acetato, 2-feniletanol, 3-metilbutanal, triptofol, tirosol, acetato de fenetilo


## 10 · Leitura dos resultados

*A preencher depois da primeira execução completa, com os números que saírem. O que abaixo está fixado é a
estrutura da leitura e os critérios, não as conclusões.*

Três perguntas, por esta ordem:

1. **Complementaridade ou competição?** O MRO alto com MIP baixo é um par que compete pelos mesmos
   nutrientes e não troca nada. Num mosto, em que o azoto assimilável é o limitante, é o resultado que se
   espera por omissão, e é o que torna a co-inoculação uma má ideia sem uma razão positiva. O MIP alto é o
   argumento a favor do par.
2. **A dependência é real?** Cada SC ≥ 0.5 tem de passar a verificação da secção 8.1. O que não passar
   fica registado como artefacto de construção do índice, não como auxotrofia.
3. **O que é que o par troca, e serve para o objetivo?** O objetivo é menos etanol com a qualidade
   sensorial mantida. Uma troca que só redistribui azoto não muda etanol nenhum. As trocas que interessam
   são as que desviam carbono da via fermentativa: lactato (*L. thermotolerans*), glicerol, ácidos
   orgânicos, e respiração parcial na *M. pulcherrima*.

## 11 · Escolha para a etapa seguinte

*A preencher com os resultados.* O critério, fixado antes de ver os números para não ser escolhido à
posteriori: leva-se para a simulação dinâmica o par com **maior MIP e menor MRO** entre os que têm
dependência confirmada por FBA, mais o par *S. cerevisiae* + *L. thermotolerans* como referência, por ser o
único com um mecanismo de redução de etanol documentado na literatura e independente do modelo (desvio de
carbono para ácido L-láctico).

## 12 · Limitações

**O que este notebook mede, e o que não mede.**

1. **O MS300 entra nos índices como lista, não como taxas.** Os bounds da equação de Sauer governam as
   leituras de FBA e não governam nenhum dos cinco índices. Uma conclusão do SMETANA neste notebook é
   sobre a composição do mosto, não sobre a sua concentração.
2. **Eixo mineral não comparável.** O yeast-GEM tem trocas separadas para K, Na, Ca, Mn, Cu, Zn e Mg; os
   CarveFungi têm um `ash 1 g` agregado. Competição e partilha de minerais entre a *S. cerevisiae* e as
   não-*Saccharomyces* são invisíveis aqui.
3. **A via de Ehrlich não está fechada em nenhum dos três CarveFungi** (sem trocas para 2-feniletanol,
   triptofol, tirosol, propanol). É a classe de interação mais relevante para aroma, e o SMETANA está cego
   a ela. Já declarado no anexo do `obj1b`; volta aqui porque é este notebook que sofre a consequência.
4. **119 órfãos.** Compostos que só os CarveFungi têm no extracelular. Enviesam qualquer comparação entre
   um par com *S. cerevisiae* e um par sem ela: o segundo tem mais moeda de troca disponível por construção
   do ficheiro, não por biologia.
5. **A NS1 é preenchimento de lacuna guiado por fenótipo.** Sem ela não há consórcio anaeróbio com a
   *T. delbrueckii* nem com a *L. thermotolerans*. O fenótipo está documentado (ambas fazem segunda
   fermentação em garrafa em espumante); o homólogo de `OSM1`/`FRD1` nos três genomas continua por
   confirmar. Todos os índices que envolvem Td e Lt em anaerobiose dependem desta edição.
6. **`min_growth` não é único.** O MIP e o MRO são viáveis em gamas disjuntas. Cada índice é reportado com o
   `min_growth` a que corre, e um índice reportado sem esse valor não é interpretável.
7. **O MIP é um MILP com óptimos alternativos** e não é determinista entre execuções.
8. **O tecto do esterol continua a ser protocolo e não mosto.** Herdado do `obj2`: o eixo do esterol não
   está modelado, e a *M. pulcherrima* não produz glicerol por falta de glicerol-3-fosfatase no ficheiro.
9. **Solver SCIP.** Todos os resultados vêm do SCIP através do `reframed`. O `obj1a` já mostrou que
   respostas binárias perto da tolerância mudam com o solver. Repetir com Gurobi antes de publicar.

## 13 · Bibliografia

1. Zelezniak, A., Andrejev, S., Ponomarova, O., Mende, D. R., Bork, P., & Patil, K. R. (2015). Metabolic
   dependencies drive species co-occurrence in diverse microbial communities. *PNAS*, 112(20), 6449–6454.
   https://doi.org/10.1073/pnas.1421834112

2. Machado, D. SMETANA: Species METabolic interaction ANAlysis (v1.2.1).
   https://github.com/cdanielmachado/smetana — implementação do MPS como índice binário e não graduado:
   issue #14, https://github.com/cdanielmachado/smetana/issues/14

3. Machado, D., Andrejev, S., Tramontano, M., & Patil, K. R. (2018). Fast automated reconstruction of
   genome-scale metabolic models for microbial species and communities. *Nucleic Acids Research*, 46(15),
   7542–7553. https://doi.org/10.1093/nar/gky537

4. Lu, H., Li, F., Sánchez, B. J., et al. (2019). A consensus *S. cerevisiae* metabolic model Yeast8 and
   its ecosystem for comprehensively probing cellular metabolism. *Nature Communications*, 10, 3586.
   https://doi.org/10.1038/s41467-019-11581-3

5. Evers, M. S., Ramousse, L., Morge, C., et al. (2023). Global mRNA and protein dynamics of
   *Saccharomyces cerevisiae* in a synthetic grape must. (Composição do MS300 usada no `obj2`, § 2.2.)

6. Varela, C., Pizarro, F., & Agosin, E. (2004). Biomass content governs fermentation rate in
   nitrogen-deficient wine musts. *Applied and Environmental Microbiology*, 70(6), 3392–3400.
   https://doi.org/10.1128/AEM.70.6.3392-3400.2004

7. Gobbi, M., Comitini, F., Domizio, P., et al. (2013). *Lachancea thermotolerans* and
   *Saccharomyces cerevisiae* in simultaneous and sequential co-fermentation: a strategy to enhance
   acidity and improve the overall quality of wine. *Food Microbiology*, 33(2), 271–281.
   https://doi.org/10.1016/j.fm.2012.10.004

8. Morales, P., Rojas, V., Quirós, M., & Gonzalez, R. (2015). The impact of oxygen on the final alcohol
   content of wine fermented by a mixed starter culture. *Applied Microbiology and Biotechnology*, 99(9),
   3993–4003. https://doi.org/10.1007/s00253-014-6321-3

9. Contreras, A., Hidalgo, C., Henschke, P. A., et al. (2014). Evaluation of non-*Saccharomyces* yeasts
   for the reduction of alcohol content in wine. *Applied and Environmental Microbiology*, 80(5),
   1670–1678. https://doi.org/10.1128/AEM.03780-13

10. Jolly, N. P., Varela, C., & Pretorius, I. S. (2014). Not your ordinary yeast: non-*Saccharomyces*
    yeasts in wine production revisited. *FEMS Yeast Research*, 14(2), 215–237.
    https://doi.org/10.1111/1567-1364.12111